# PyTorch: Important Concepts from Basic to Advanced

---

## Overview

This notebook includes **essential PyTorch concepts** for deep learning: tensors, operations, automatic differentiation, building models with `nn.Module`, optimization, data loading, and a complete training pipeline. By the end you will be able to write and train neural networks in PyTorch from scratch.

**Prerequisites:** Basic Python and NumPy.  
**Estimated time:** 3–5 hours  

---

### Table of Contents

1. **Setup and imports**  
2. **Tensors** — creation, indexing, dtype, device  
3. **Operations and broadcasting**  
4. **Automatic differentiation (autograd)**  
5. **Building models with `nn.Module`**  
6. **Loss functions and optimizers**  
7. **Datasets and DataLoader**  
8. **Full training loop**  
9. **Saving and loading models**  
10. **Summary and practice**

---

In [2]:
# !pip install torch torchvision torchaudio

## 1. Setup and Imports

Install PyTorch if needed: `pip install torch` (see [pytorch.org](https://pytorch.org) for CUDA versions). We use `torch` and `torch.nn` for building and training models.

In [3]:
import torch
import torch.nn as nn
import numpy as np

# Reproducibility 
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)

# Check PyTorch version and whether accelerated device is available
print("PyTorch version:", torch.__version__)

# Check MPS (Mac) first, then CUDA, fallback to CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

# Print device details
if device.type == "mps":
    print("Accelerator: Apple Metal Performance Shaders (MPS)")
elif device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch version: 2.10.0+cpu
Device: cpu


---

## 2. Tensors: The Core Data Structure

A **tensor** is PyTorch's multi-dimensional array (like NumPy's `ndarray`). Tensors can live on CPU or GPU and support **automatic differentiation** for training neural networks.

**Key ideas:**
- Shape: `(batch, channels, height, width)` for images; `(batch, features)` for vectors.
- **dtype:** `float32` is default for learning; `long` (int64) for indices/labels.
- **device:** `cpu` or `cuda` or `mps`; all operands in an op must be on the same device.

### 2.1 Creating tensors

From lists, zeros, ones, ranges, random values, or from NumPy.

In [4]:
# From Python list
a = torch.tensor([[1., 2.], [3., 4.]])
print("From list:", a)

# Zeros, ones, with shape (rows, cols)
z = torch.zeros(2, 3)
o = torch.ones(2, 3)
print("Zeros shape:", z.shape)

# Range: torch.arange(start, end, step)
r = torch.arange(0, 10, 2)
print("Arange:", r)

# Random (uniform in [0,1), or randn for standard normal)
rand = torch.rand(2, 3)
randn = torch.randn(2, 3)
print("Rand (uniform) shape:", rand.shape)

# From NumPy (shares memory if possible when on CPU)
np_arr = np.array([1, 2, 3], dtype=np.float32)
t = torch.from_numpy(np_arr)
print("From NumPy:", t)

From list: tensor([[1., 2.],
        [3., 4.]])
Zeros shape: torch.Size([2, 3])
Arange: tensor([0, 2, 4, 6, 8])
Rand (uniform) shape: torch.Size([2, 3])
From NumPy: tensor([1., 2., 3.])


### 2.2 Tensor attributes: shape, dtype, device

Always check shape when debugging; use the right dtype (float32 for parameters, long for class indices).

In [5]:
x = torch.randn(4, 5)
print("Shape:", x.shape)        # or x.size()
print("Number of elements:", x.numel())
print("dtype:", x.dtype)
print("device:", x.device)

# Create on GPU (if available)
# Create on accelerated device (MPS/CUDA/CPU)
x_accelerated = x.to(device)
print("On device:", x_accelerated.device)


# Specify dtype at creation
labels = torch.tensor([0, 1, 2], dtype=torch.long)  # for class indices
print("Labels dtype:", labels.dtype)

Shape: torch.Size([4, 5])
Number of elements: 20
dtype: torch.float32
device: cpu
On device: cpu
Labels dtype: torch.int64


### 2.3 Indexing and slicing

Same rules as NumPy: indexing, slicing, boolean masks. Use `.item()` to get a single Python scalar from a one-element tensor.

In [6]:
x = torch.tensor([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]])
print("x[1, 2]:", x[1, 2])           # row 1, col 2
print("x[:, 0]:", x[:, 0])           # first column
print("x[1:3, :]:", x[1:3, :])       # rows 1 and 2

# Single-element tensor -> Python scalar
val = x[0, 0].item()
print("As Python float:", val, type(val))

x[1, 2]: tensor(6.)
x[:, 0]: tensor([1., 4., 7.])
x[1:3, :]: tensor([[4., 5., 6.],
        [7., 8., 9.]])
As Python float: 1.0 <class 'float'>


### 2.4 Reshaping: view, reshape, squeeze, unsqueeze

- **view:** New shape must have same total number of elements; shares storage with original. Fast.
- **reshape:** Like view but may return a copy if contiguous. Prefer view when you know layout.
- **flatten:** Collapse all dimensions (e.g. (2,3,4) -> (24,)).
- **squeeze / unsqueeze:** Remove or add dimensions of size 1 (e.g. (3,1,4) -> (3,4)).

In [7]:
x = torch.arange(12)
print("Original:", x.shape)

# Reshape to (3, 4)
y = x.view(3, 4)
print("view(3, 4):", y.shape)

# Flatten to 1D
z = y.flatten()
print("flatten:", z.shape)

# Add batch dimension: (3, 4) -> (1, 3, 4) for a single sample
batch = y.unsqueeze(0)
print("unsqueeze(0):", batch.shape)

# Remove size-1 dims
out = batch.squeeze()
print("squeeze:", out.shape)

Original: torch.Size([12])
view(3, 4): torch.Size([3, 4])
flatten: torch.Size([12])
unsqueeze(0): torch.Size([1, 3, 4])
squeeze: torch.Size([3, 4])


---

## 3. Operations and Broadcasting

PyTorch supports element-wise math, matrix multiply, and **broadcasting** (like NumPy): dimensions are aligned from the right; missing dimensions are treated as size 1 and "stretched."

In [8]:
a = torch.tensor([1., 2., 3.])
b = torch.tensor([10., 20., 30.])

print("a + b:", a + b)
print("a * b:", a * b)
print("a ** 2:", a ** 2)

# Matrix multiplication: (m,n) @ (n,p) -> (m,p)
A = torch.randn(2, 3)
B = torch.randn(3, 4)
C = A @ B   # or torch.mm(A, B) for 2D
print("A @ B shape:", C.shape)

# Broadcasting: (3,) + (3, 1) -> (3, 3)
v = torch.tensor([1., 2., 3.])
w = torch.tensor([[10.], [20.], [30.]])
print("v + w (broadcast):", (v + w))

a + b: tensor([11., 22., 33.])
a * b: tensor([10., 40., 90.])
a ** 2: tensor([1., 4., 9.])
A @ B shape: torch.Size([2, 4])
v + w (broadcast): tensor([[11., 12., 13.],
        [21., 22., 23.],
        [31., 32., 33.]])


### In-place vs out-of-place

Operations like `x.add_(y)` modify `x` in place (suffix `_`). Use sparingly; autograd can be tricky with in-place ops on tensors that require gradients.

In [9]:
x = torch.tensor([1., 2., 3.])
y = x.add(10)   # new tensor; x unchanged
print("x after add:", x)

x.add_(10)      # in-place; x is modified
print("x after add_:", x)

x after add: tensor([1., 2., 3.])
x after add_: tensor([11., 12., 13.])


---

## 4. Automatic Differentiation (Autograd)

PyTorch records operations on tensors that have `requires_grad=True`. When you call `.backward()` on a scalar loss, it computes gradients for all tensors that took part in the computation. These gradients are used by optimizers to update parameters.

**Important:**
- Set `requires_grad=True` on parameters you want to learn.
- Loss must be a **scalar** for `loss.backward()`.
- Call `optimizer.zero_grad()` before each backward pass to clear old gradients.

### 4.1 Simple gradient example

We define a simple function and compute the gradient of its output w.r.t. the input.

In [10]:
# Create a tensor that will track gradients
x = torch.tensor([2.0], requires_grad=True)

# y = x^2 -> dy/dx = 2x
y = x ** 2
y.backward()   # compute gradients

print("x.grad:", x.grad)   # should be 2*2 = 4
print("Expected dy/dx = 2x at x=2:", 2 * 2)

x.grad: tensor([4.])
Expected dy/dx = 2x at x=2: 4


In [11]:
# Multiple inputs: z = x*y + x^2
x = torch.tensor(1.0, requires_grad=True)
y = torch.tensor(2.0, requires_grad=True)
z = x * y + x ** 2
z.backward()
print("dz/dx:", x.grad)   # y + 2x = 2 + 2 = 4
print("dz/dy:", y.grad)   # x = 1

dz/dx: tensor(4.)
dz/dy: tensor(1.)


### 4.2 Why we need zero_grad

Gradients **accumulate** by default. So we call `optimizer.zero_grad()` (or `model.zero_grad()`) at the start of each training step so we only use gradients from the current batch.

In [12]:
w = torch.tensor([1.0], requires_grad=True)
loss1 = w * 2
loss1.backward()
print("After first backward, w.grad:", w.grad)

loss2 = w * 3
loss2.backward()
print("After second backward (accumulated), w.grad:", w.grad)

w.grad.zero_()
print("After zero_grad:", w.grad)

After first backward, w.grad: tensor([2.])
After second backward (accumulated), w.grad: tensor([5.])
After zero_grad: tensor([0.])


---

## 5. Building Models with `nn.Module`

All trainable models in PyTorch subclass **`nn.Module`**. You define layers in `__init__` and the forward pass in **`forward(self, x)`**. Parameters (weights and biases) are registered automatically when you assign `nn.Linear`, `nn.Conv2d`, etc. as attributes.

- **`model.parameters()`:** Iterator over all learnable parameters.  
- **`model.to(device)`:** Move model (and its parameters) to CPU or GPU.

### 5.1 A simple linear model

One linear layer: `y = xW^T + b`. Input size 4, output size 2.

In [13]:
class SimpleLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.linear(x)

model = SimpleLinear(4, 2)
print(model)
print("Parameters:", list(model.parameters()))

# Forward pass: batch of 3 samples, 4 features each
x = torch.randn(3, 4)
out = model(x)
print("Input shape:", x.shape, "-> Output shape:", out.shape)

SimpleLinear(
  (linear): Linear(in_features=4, out_features=2, bias=True)
)
Parameters: [Parameter containing:
tensor([[ 0.1057, -0.1275,  0.2980,  0.3399],
        [-0.3626, -0.2669,  0.4578, -0.1687]], requires_grad=True), Parameter containing:
tensor([-0.1773, -0.4838], requires_grad=True)]
Input shape: torch.Size([3, 4]) -> Output shape: torch.Size([3, 2])


### 5.2 A small MLP (multi-layer perceptron)

Stack of Linear + ReLU, then a final Linear for class scores. Use `nn.Sequential` to chain layers.

In [14]:
class SmallMLP(nn.Module):
    def __init__(self, input_size=28*28, hidden_size=128, num_classes=10):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, num_classes),
        )

    def forward(self, x):
        # x might be (batch, 1, 28, 28); flatten to (batch, 784)
        x = x.view(x.size(0), -1)
        return self.layers(x)

model = SmallMLP()
print("Total parameters:", sum(p.numel() for p in model.parameters()))

x = torch.randn(2, 1, 28, 28)
print("Output shape:", model(x).shape)

Total parameters: 118282
Output shape: torch.Size([2, 10])


### 5.3 model.train() and model.eval()

**Training mode** (`model.train()`): Dropout and BatchNorm behave for training (e.g. dropout is active).  
**Evaluation mode** (`model.eval()`): Dropout is off, BatchNorm uses running statistics. Always use `model.eval()` when evaluating or doing inference.

In [15]:
model = SmallMLP()
model.train()
x = torch.randn(1, 784)
out1 = model(x)

model.eval()
with torch.no_grad():   # no need to store gradients for inference
    out2 = model(x)

print("Train mode output (may vary if dropout):", out1[0, :3])
print("Eval mode output:", out2[0, :3])

Train mode output (may vary if dropout): tensor([0.0115, 0.1051, 0.0250], grad_fn=<SliceBackward0>)
Eval mode output: tensor([0.0115, 0.1051, 0.0250])


---

## 6. Loss Functions and Optimizers

- **Loss:** Measures how wrong the prediction is. For classification we use **cross-entropy** (combines log-softmax and NLL). For regression, MSE or L1.  
- **Optimizer:** Updates parameters using gradients. **SGD** and **Adam** are the most common. You pass `model.parameters()` and a learning rate.

In [16]:
import torch.optim as optim

model = SmallMLP()

# Cross-entropy for classification: logits (no softmax) and class indices
criterion = nn.CrossEntropyLoss()

# Optimizer: Adam is a good default
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# One step: forward -> loss -> backward -> step
x = torch.randn(4, 784)
labels = torch.tensor([0, 1, 2, 0], dtype=torch.long)

optimizer.zero_grad()
logits = model(x)
loss = criterion(logits, labels)
loss.backward()
optimizer.step()

print("Loss:", loss.item())

Loss: 2.373753547668457


### Common loss functions

| Task        | Loss                  | Notes                                      |
|-------------|-----------------------|--------------------------------------------|
| Classification | `nn.CrossEntropyLoss()` | Logits + class indices (long tensor)     |
| Regression  | `nn.MSELoss()`        | Mean squared error                         |
| Binary      | `nn.BCEWithLogitsLoss()` | Logits; targets in [0,1]                |

---

## 7. Datasets and DataLoader

**Dataset:** Implements `__len__` and `__getitem__(idx)` to return one sample (e.g. image, label).  
**DataLoader:** Batches samples, shuffles (for training), and can use multiple workers to load data in parallel.

We use the built-in **MNIST** dataset and apply a simple transform (ToTensor, Normalize).

In [17]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))   # MNIST mean and std
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print("Train samples:", len(train_dataset))
print("Batches per epoch:", len(train_loader))

# One batch
images, labels = next(iter(train_loader))
print("Batch images shape:", images.shape)
print("Batch labels shape:", labels.shape)

100.0%
100.0%
100.0%
100.0%

Train samples: 60000
Batches per epoch: 938
Batch images shape: torch.Size([64, 1, 28, 28])
Batch labels shape: torch.Size([64])


### Custom Dataset (skeleton)

For your own data, subclass `Dataset` and implement `__len__` and `__getitem__`. Return (input_tensor, label) or (input_tensor,) for unsupervised data.

In [18]:
from torch.utils.data import Dataset

class DummyDataset(Dataset):
    def __init__(self, num_samples=100, input_dim=10, num_classes=3):
        self.X = torch.randn(num_samples, input_dim)
        self.y = torch.randint(0, num_classes, (num_samples,))

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

ds = DummyDataset(50)
loader = DataLoader(ds, batch_size=10, shuffle=True)
xb, yb = next(iter(loader))
print("Custom batch:", xb.shape, yb.shape)

Custom batch: torch.Size([10, 10]) torch.Size([10])


---

## 8. Full Training Loop

Putting it all together: for each epoch we iterate over the train DataLoader, do forward pass, compute loss, backward, and optimizer step. We then evaluate on the test set (with `model.eval()` and `torch.no_grad()`).

In [19]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            pred = logits.argmax(dim=1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
    return correct / total

In [20]:
model = SmallMLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 3
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_acc = evaluate(model, test_loader, device)
    print(f"Epoch {epoch+1}/{num_epochs}  Train loss: {train_loss:.4f}  Train acc: {train_acc:.4f}  Test acc: {test_acc:.4f}")

Epoch 1/3  Train loss: 0.2566  Train acc: 0.9250  Test acc: 0.9606
Epoch 2/3  Train loss: 0.1070  Train acc: 0.9671  Test acc: 0.9675
Epoch 3/3  Train loss: 0.0770  Train acc: 0.9765  Test acc: 0.9746


---

## 9. Saving and Loading Models

**Save:** `torch.save(model.state_dict(), path)` saves only the parameters (recommended).  
**Load:** Create the same model architecture, then `model.load_state_dict(torch.load(path))`.  
Use `model.eval()` after loading when you want to run inference.

In [21]:
# Save
path = "mnist_mlp.pt"
torch.save(model.state_dict(), path)
print("Saved to", path)

# Load: same architecture, then load weights
model_loaded = SmallMLP().to(device)
model_loaded.load_state_dict(torch.load(path, map_location=device))
model_loaded.eval()
print("Loaded. Test acc:", evaluate(model_loaded, test_loader, device))

Saved to mnist_mlp.pt
Loaded. Test acc: 0.9746


---

## 10. Summary and Practice

**Concepts covered:**
- **Tensors:** creation, shape, dtype, device, indexing, view/reshape.  
- **Autograd:** `requires_grad`, `.backward()`, `.zero_grad()`.  
- **nn.Module:** `__init__` (layers), `forward`, `parameters()`, `train()`/`eval()`.  
- **Training:** Loss (e.g. CrossEntropyLoss), optimizer (e.g. Adam), loop: zero_grad → forward → loss → backward → step.  
- **Data:** Dataset, DataLoader, move batch to device.  
- **Persistence:** `state_dict`, save/load.

**Practice ideas:**  
1. Change the MLP hidden size and number of layers; compare accuracy.  
2. Try SGD instead of Adam; tune learning rate.  
3. Implement a validation split and print validation accuracy each epoch.  
4. Save the best model by validation accuracy.  
5. Use a different dataset (e.g. CIFAR-10 with a small CNN).

---

**Quick reference:**  
- `x.to(device)` — move tensor to GPU/CPU  
- `model(x)` — forward pass (calls `forward`)  
- `loss.backward()` — compute gradients  
- `optimizer.step()` — update parameters  
- `with torch.no_grad():` — disable gradient tracking (inference)